# Myanmar Sign Language (MSL) Recognition for Emergency Domain - Version 2.0
## Runtime Environment: Kaggle (T4 GPU)

## Setup

### Path Setup

In [2]:
# copy files in dataset dir to working dir of a session
import shutil
import os

src = "/kaggle/input/datasets/lawunnannda/msl4emergency-dataset-augmented-keypoints"
dst = "/kaggle/working/"

shutil.copytree(src, dst, dirs_exist_ok=True)

'/kaggle/working/'

In [3]:
%pwd

'/kaggle/working'

In [5]:
!ls

annotations.txt  config     label_map.json  sample4infer  src	    wandb
augmented	 keypoints  results	    scripts	  state.db


In [6]:
# create data/ and move files to match paths in config
!mkdir data/
!mv augmented/ sample4infer/ annotations.txt label_map.json data/

In [7]:
!ls

config	data  keypoints  results  scripts  src	state.db  wandb


In [17]:
from pathlib import Path

ROOT = Path(dst).resolve()
KP_DIR = ROOT / "data" / "keypoints"

CONFIG = ROOT / "config" / "config.yaml"
AUG_DIR = ROOT / "data" / "augmented"
SCRIPTS_DIR = ROOT / "scripts"
SRC_DIR = ROOT / "src"
RESULTS_DIR = ROOT / "results"

DATA_ROOT = Path(src).resolve()
REAL_KP_DIR = DATA_ROOT / "keypoints"

In [9]:
# symlink for keypoints
!ln -sfn {REAL_KP_DIR} {KP_DIR}

# verify a symlink
!ls -la {KP_DIR}

lrwxrwxrwx 1 root root 86 Jun 22 08:37 /kaggle/working/data/keypoints -> /kaggle/input/datasets/lawunnannda/msl4emergency-dataset-augmented-keypoints/keypoints


In [10]:
# final check
%cd {ROOT}
!echo
!ls
!echo
!ls {ROOT}/"data"

/kaggle/working

config	data  keypoints  results  scripts  src	state.db  wandb

annotations.txt  augmented  keypoints  label_map.json  sample4infer


In [8]:
# check scripts
!ls -l {SCRIPTS_DIR}

total 60
-rw-r--r-- 1 root root 3868 Jun 21 22:59 00_setup_env.sh
-rw-r--r-- 1 root root 2999 Jun 21 22:59 01_prepare_data.sh
-rw-r--r-- 1 root root 2695 Jun 21 22:59 02_extract_keypoints.sh
-rw-r--r-- 1 root root 2971 Jun 21 22:59 03_augment_data.sh
-rw-r--r-- 1 root root 3162 Jun 21 22:59 04_train.sh
-rw-r--r-- 1 root root 2696 Jun 21 22:59 05_evaluate.sh
-rw-r--r-- 1 root root 4880 Jun 21 22:59 06_run_all_experiments.sh
-rw-r--r-- 1 root root 3310 Jun 21 22:59 07_cross_validate.sh
-rw-r--r-- 1 root root 2625 Jun 21 22:59 08_infer.sh
-rw-r--r-- 1 root root  642 Jun 21 22:59 09_tensorboard.sh
-rw-r--r-- 1 root root 2160 Jun 21 22:59 compare_results.py
-rw-r--r-- 1 root root 5709 Jun 21 22:59 error_analysis.py
-rw-r--r-- 1 root root  725 Jun 21 22:59 move_videos.sh


### Library Setup

In [13]:
import numpy as np
import os
import wandb
import kagglehub

### Weights & Biases (Wandb) Setup

In [14]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

In [15]:
!wandb login --relogin {wandb_api}

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


### Files Setup

In [12]:
# check augmented data
%cd {ROOT}
!ls "{AUG_DIR}"

/kaggle/working
augmented_manifest.json  train	val


In [13]:
# check manifest file
!cat "{AUG_DIR}/augmented_manifest.json" | tail -n 20

      "normal_text": "ဟင့်အင်း ၊ သတိ မ လစ် ပါ ဘူး ။",
      "msl_gloss": "သတိလစ် မဟုတ်ဘူး သတိ ရတယ်",
      "label": "သတိလစ် မဟုတ်ဘူး သတိ ရတယ်",
      "video_path": "data/videos/501-558/idx20-557.mp4",
      "keypoint_path": "data/keypoints/501-558/idx20-557.npy",
      "is_augmented": false,
      "split": "test"
    },
    {
      "idx": 557,
      "normal_text": "ဆရာဝန် ဘယ် အချိန် လာ မှာ လဲ ။",
      "msl_gloss": "ဆရာဝန် လာ အချိန် ဘယ်လောက်လဲ",
      "label": "ဆရာဝန် လာ အချိန် ဘယ်လောက်လဲ",
      "video_path": "data/videos/501-558/idx20-558.mp4",
      "keypoint_path": "data/keypoints/501-558/idx20-558.npy",
      "is_augmented": false,
      "split": "test"
    }
  ]
}

In [14]:
# check augmented training data
!ls "{AUG_DIR}/train/" | head -n 5

0000_aug001.npy
0000_aug002.npy
0000_aug003.npy
0000_aug004.npy
0000_aug005.npy
ls: write error: Broken pipe


In [15]:
# check augmented validation data
!ls "{AUG_DIR}/val/" | head -n 5

0000_aug000.npy
0001_aug000.npy
0002_aug000.npy
0003_aug000.npy
0004_aug000.npy


## 7. Training with Cross Validation

In [16]:
!cat {CONFIG} | head -n 12

# ============================================================
# Myanmar Sign Language Recognition - Configuration
# ============================================================

data:
  video_dir: "data/videos"
  keypoint_dir: "data/keypoints"
  augmented_dir: "data/augmented"
  annotation_file: "data/annotations.txt"
  label_map_file: "data/label_map.json"
  split_file: "data/splits.json"



### BiLSTM Model

In [20]:
!time bash {SCRIPTS_DIR}/07_cross_validate.sh bilstm exp_cv_bilstm 5

 K-Fold Cross-Validation
 Model  : bilstm
 Exp    : exp_cv_bilstm
 Folds  : 5
 W&B    : yes

 Aug train pool : 10602 samples (all classes, folded into 5 folds)
 Test set       : 558 originals (fixed across all folds)

2026-06-21 23:11:30.309556: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782083490.519568     406 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782083490.582153     406 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782083491.116529     406 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782083491.116591     406 comput

In [11]:
shutil.make_archive(
    f"{RESULTS_DIR}/exp_cv_bilstm",
    'zip',
    f"{RESULTS_DIR}/exp_cv_bilstm"
)

'/kaggle/working/results/exp_cv_bilstm.zip'

### Transformer Model

In [16]:
!time bash {SCRIPTS_DIR}/07_cross_validate.sh transformer exp_cv_transformer 5

 K-Fold Cross-Validation
 Model  : transformer
 Exp    : exp_cv_transformer
 Folds  : 5
 W&B    : yes

 Aug train pool : 10602 samples (all classes, folded into 5 folds)
 Test set       : 558 originals (fixed across all folds)

2026-06-22 08:50:53.187822: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782118253.211635     236 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782118253.219551     236 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782118253.239980     236 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782118253.240007     

In [21]:
shutil.make_archive(
    f"{RESULTS_DIR}/exp_cv_transformer",
    'zip',
    f"{RESULTS_DIR}/exp_cv_transformer"
)

'/kaggle/working/results/exp_cv_transformer.zip'

### Spatial-Temporal Graph Convolutional Network (ST-GCN) Model

In [ ]:
!time bash {SCRIPTS_DIR}/07_cross_validate.sh stgcn exp_cv_stgcn 5

In [ ]:
shutil.make_archive(
    "/kaggle/working/results/exp_cv_stgcn",
    'zip',
    "/kaggle/working/results/exp_cv_stgcn"
)

## 5. Evaluate

In [ ]:
!python {SRC_DIR}/plot_results_v2.py \
  --exp results/exp_cv_bilstm/fold_00 \
       results/exp_cv_bilstm/fold_01 \
       results/exp_cv_bilstm/fold_02 \
       results/exp_cv_bilstm/fold_03 \
       results/exp_cv_bilstm/fold_04